In [1]:
import cv2
import os
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from collections import defaultdict
from ultralytics import YOLO
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import json
import cv2
import numpy as np
import urllib.request
import json

In [2]:
cap = cv2.VideoCapture("output.mp4")
os.makedirs("frames", exist_ok=True)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imwrite(f"frames/{frame_id:05d}.jpg", frame)
    frame_id += 1
cap.release()

print(f"Извлечено кадров: {frame_id}")

Извлечено кадров: 301


In [3]:
def parse_cvat_xml(xml_path):
    """
    Возвращает dict: {frame_id: [{"label": ..., "bbox": [x, y, w, h]}, ...]}
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt = defaultdict(list)

    for track in root.findall("track"):
        label = track.get("label")  # "car" или "minivan"

        for box in track.findall("box"):
            frame = int(box.get("frame"))
            outside = int(box.get("outside"))

            if outside == 1:
                continue  # объект вне кадра — пропускаем

            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))

            x = xtl
            y = ytl
            w = xbr - xtl
            h = ybr - ytl

            gt[frame].append({
                "label": label,
                "bbox": [x, y, w, h]
            })

    return gt

# Использование
gt_annotations = parse_cvat_xml("annotations.xml")

# Проверка
print(f"Кадров с разметкой: {len(gt_annotations)}")
print(f"Пример кадр 50: {gt_annotations[50]}")

Кадров с разметкой: 901
Пример кадр 50: [{'label': 'car', 'bbox': [596.63, 10.5, 30.269999999999982, 28.4]}, {'label': 'car', 'bbox': [1544.5, 506.91, 211.47000000000003, 190.69]}, {'label': 'car', 'bbox': [596.8, 0.0, 26.200000000000045, 17.8]}, {'label': 'car', 'bbox': [207.2, 753.18, 284.99, 326.82000000000005]}, {'label': 'car', 'bbox': [344.2, 394.17, 150.27000000000004, 155.22999999999996]}, {'label': 'car', 'bbox': [433.42, 126.04, 56.24000000000001, 62.08999999999999]}, {'label': 'car', 'bbox': [536.0, 159.03, 60.50999999999999, 58.97]}, {'label': 'car', 'bbox': [612.84, 72.13, 43.45999999999992, 41.07000000000001]}, {'label': 'car', 'bbox': [862.07, 364.35, 104.32999999999993, 139.95]}, {'label': 'car', 'bbox': [895.61, 533.27, 152.39, 216.02999999999997]}]


In [4]:
# Посмотрим что за кадры > 300
frames_in_gt = sorted(gt_annotations.keys())
print(f"Мин кадр: {frames_in_gt[0]}, Макс кадр: {frames_in_gt[-1]}")
print(f"Кадров в диапазоне 0-300: {sum(1 for f in frames_in_gt if f <= 300)}")
print(f"Кадров за пределами 0-300: {sum(1 for f in frames_in_gt if f > 300)}")

Мин кадр: 0, Макс кадр: 900
Кадров в диапазоне 0-300: 301
Кадров за пределами 0-300: 600


In [5]:
def parse_cvat_xml(xml_path, max_frame=300):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    gt = defaultdict(list)

    for track in root.findall("track"):
        label = track.get("label")

        for box in track.findall("box"):
            frame = int(box.get("frame"))
            outside = int(box.get("outside"))

            if outside == 1:
                continue
            if frame > max_frame:  # фильтр
                continue

            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))

            gt[frame].append({
                "label": label,
                "bbox": [xtl, ytl, xbr - xtl, ybr - ytl]
            })

    return gt

gt_annotations = parse_cvat_xml("annotations.xml", max_frame=300)
print(f"Кадров с разметкой: {len(gt_annotations)}")

Кадров с разметкой: 301


In [8]:
def extract_bboxes_from_frame(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    color_ranges = {
        "blue":   ([100, 150, 100], [130, 255, 255]),
        "cyan":   ([85,  100, 100], [100, 255, 255]),
        # green убрали — ловит деревья
        "red1":   ([0,   150, 100], [10,  255, 255]),
        "red2":   ([170, 150, 100], [180, 255, 255]),
        "yellow": ([20,  150, 100], [35,  255, 255]),
        "pink":   ([140, 100, 100], [165, 255, 255]),
    }

    combined_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    for lower, upper in color_ranges.values():
        mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
        combined_mask = cv2.bitwise_or(combined_mask, mask)

    kernel = np.ones((3, 3), np.uint8)
    combined_mask = cv2.dilate(combined_mask, kernel, iterations=2)

    contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    bboxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        area = w * h
        aspect = w / h if h > 0 else 0

        if area < 1500:
            continue
        if aspect < 0.3 or aspect > 5:
            continue

        # Проверка прямоугольности: площадь контура vs площадь bbox
        # У деревьев контур "рваный", у bbox — почти прямоугольник
        contour_area = cv2.contourArea(c)
        rect_fill = contour_area / area if area > 0 else 0
        if rect_fill < 0.25:  # контур заполняет меньше 25% bbox → не прямоугольник
            continue

        bboxes.append([x, y, w, h])

    return bboxes

# Тест
frame = cv2.imread("frames/00000.jpg")
bboxes = extract_bboxes_from_frame(frame)
print(f"Найдено bbox'ов: {len(bboxes)}")

for (x, y, w, h) in bboxes:
    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
cv2.imwrite("debug_frame2.jpg", frame)

Найдено bbox'ов: 12


True

In [9]:
cap_in = cv2.VideoCapture("input.mp4")
cap_out = cv2.VideoCapture("output.mp4")

def extract_bboxes_diff(frame_out, frame_in):
    # Разность кадров — остаются только нарисованные bbox'и
    diff = cv2.absdiff(frame_out, frame_in)

    # Переводим в серый и бинаризуем
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)

    # Морфология — убираем шум
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    mask = cv2.erode(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    bboxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        area = w * h
        aspect = w / h if h > 0 else 0

        if area < 1500:
            continue
        if aspect < 0.3 or aspect > 5:
            continue

        bboxes.append([x, y, w, h])

    return bboxes

# Тест на первом кадре
cap_in.set(cv2.CAP_PROP_POS_FRAMES, 0)
cap_out.set(cv2.CAP_PROP_POS_FRAMES, 0)

_, frame_in  = cap_in.read()
_, frame_out = cap_out.read()

bboxes = extract_bboxes_diff(frame_out, frame_in)
print(f"Найдено bbox'ов: {len(bboxes)}")

# Визуализация
debug = frame_out.copy()
for (x, y, w, h) in bboxes:
    cv2.rectangle(debug, (x, y), (x+w, y+h), (0, 255, 0), 2)
cv2.imwrite("debug_diff.jpg", debug)

cap_in.release()
cap_out.release()

Найдено bbox'ов: 7


In [10]:
cap_in = cv2.VideoCapture("input.mp4")
cap_out = cv2.VideoCapture("output.mp4")

all_bboxes = {}  # {frame_id: [[x,y,w,h], ...]}

total_frames = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))

for frame_id in range(total_frames):
    ret_in,  frame_in  = cap_in.read()
    ret_out, frame_out = cap_out.read()

    if not ret_in or not ret_out:
        break

    bboxes = extract_bboxes_diff(frame_out, frame_in)
    all_bboxes[frame_id] = bboxes

    if frame_id % 50 == 0:
        print(f"Кадр {frame_id}/{total_frames}, найдено bbox'ов: {len(bboxes)}")

cap_in.release()
cap_out.release()
print(f"Готово. Обработано кадров: {len(all_bboxes)}")

Кадр 0/301, найдено bbox'ов: 7
Кадр 50/301, найдено bbox'ов: 9
Кадр 100/301, найдено bbox'ов: 6
Кадр 150/301, найдено bbox'ов: 6
Кадр 200/301, найдено bbox'ов: 10
Кадр 250/301, найдено bbox'ов: 10
Кадр 300/301, найдено bbox'ов: 13
Готово. Обработано кадров: 301


In [12]:
def iou(boxA, boxB):
    # [x, y, w, h] → [x1, y1, x2, y2]
    ax1, ay1 = boxA[0], boxA[1]
    ax2, ay2 = ax1 + boxA[2], ay1 + boxA[3]
    bx1, by1 = boxB[0], boxB[1]
    bx2, by2 = bx1 + boxB[2], by1 + boxB[3]

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = boxA[2]*boxA[3] + boxB[2]*boxB[3] - inter
    return inter / union if union > 0 else 0.0

def match_bboxes(pred_boxes, gt_boxes, iou_thresh=0.5):
    """
    Для каждого gt bbox находим лучший pred bbox по IoU.
    Возвращает список iou для matched пар.
    """
    matched_ious = []
    used_preds = set()

    for gt in gt_boxes:
        best_iou = 0
        best_idx = -1
        for i, pred in enumerate(pred_boxes):
            if i in used_preds:
                continue
            score = iou(pred, gt)
            if score > best_iou:
                best_iou = score
                best_idx = i

        if best_idx >= 0 and best_iou >= iou_thresh:
            matched_ious.append(best_iou)
            used_preds.add(best_idx)
        else:
            matched_ious.append(0.0)  # не нашли пару → IoU = 0

    return matched_ious

# Считаем IoU по всем кадрам
all_ious = []

for frame_id in range(301):
    pred_boxes = all_bboxes.get(frame_id, [])
    gt_boxes = [ann["bbox"] for ann in gt_annotations.get(frame_id, [])]

    if not gt_boxes:
        continue

    frame_ious = match_bboxes(pred_boxes, gt_boxes, iou_thresh=0.3)
    all_ious.extend(frame_ious)

mean_iou = sum(all_ious) / len(all_ious) if all_ious else 0
matched = sum(1 for x in all_ious if x > 0)

print(f"Всего gt bbox'ов: {len(all_ious)}")
print(f"Matched (IoU > 0.3): {matched} ({100*matched/len(all_ious):.1f}%)")
print(f"Mean IoU: {mean_iou:.4f}")

Всего gt bbox'ов: 3647
Matched (IoU > 0.3): 2058 (56.4%)
Mean IoU: 0.3436


In [15]:
yolo = YOLO("yolov8n.pt")  # скачается автоматически

def extract_bboxes_yolo(frame):
    results = yolo(frame, verbose=False)[0]
    bboxes = []
    for box in results.boxes:
        cls = int(box.cls[0])
        if cls != 2:  # 2 = car в COCO классах
            continue
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        bboxes.append([int(x1), int(y1), int(x2-x1), int(y2-y1)])
    return bboxes

# Прогон по всем кадрам
cap = cv2.VideoCapture("input.mp4")  # берём чистое видео
all_bboxes_yolo = {}

for frame_id in range(301):
    ret, frame = cap.read()
    if not ret:
        break
    all_bboxes_yolo[frame_id] = extract_bboxes_yolo(frame)
    if frame_id % 50 == 0:
        print(f"Кадр {frame_id}/301, bbox'ов: {len(all_bboxes_yolo[frame_id])}")

cap.release()

# IoU для YOLO
all_ious_yolo = []
for frame_id in range(301):
    pred_boxes = all_bboxes_yolo.get(frame_id, [])
    gt_boxes = [ann["bbox"] for ann in gt_annotations.get(frame_id, [])]
    if not gt_boxes:
        continue
    all_ious_yolo.extend(match_bboxes(pred_boxes, gt_boxes, iou_thresh=0.3))

mean_iou_yolo = sum(all_ious_yolo) / len(all_ious_yolo)
matched_yolo = sum(1 for x in all_ious_yolo if x > 0)

print(f"\n=== YOLO ===")
print(f"Matched: {matched_yolo} ({100*matched_yolo/len(all_ious_yolo):.1f}%)")
print(f"Mean IoU: {mean_iou_yolo:.4f}")

print(f"\n=== Сравнение методов ===")
print(f"Diff:  matched={56.4}%, IoU={0.3436:.4f}")
print(f"YOLO:  matched={100*matched_yolo/len(all_ious_yolo):.1f}%, IoU={mean_iou_yolo:.4f}")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\KatyaSigma\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Кадр 0/301, bbox'ов: 6
Кадр 50/301, bbox'ов: 5
Кадр 100/301, bbox'ов: 6
Кадр 150/301, bbox'ов: 6
Кадр 200/301, bbox'ов: 10
Кадр 250/301, bbox'ов: 10
Кадр 300/301, bbox'ов: 10

=== YOLO ===
Matched: 2125 (58.3%)
Mean IoU: 0.4932

=== Сравнение методов ===
Diff:  matched=56.4%, IoU=0.3436
YOLO:  matched=58.3%, IoU=0.4932


In [18]:
import urllib.request

# Скачиваем каскад для машин
url = "https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml"
urllib.request.urlretrieve(url, "haarcascade_car.xml")

car_cascade = cv2.CascadeClassifier("haarcascade_car.xml")

def extract_bboxes_haar(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cars = car_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=3,
        minSize=(30, 30)
    )
    if len(cars) == 0:
        return []
    return [[x, y, w, h] for (x, y, w, h) in cars]

# Прогон по всем кадрам
cap = cv2.VideoCapture("input.mp4")
all_bboxes_haar = {}

for frame_id in range(301):
    ret, frame = cap.read()
    if not ret:
        break
    all_bboxes_haar[frame_id] = extract_bboxes_haar(frame)
    if frame_id % 50 == 0:
        print(f"Кадр {frame_id}/301, bbox'ов: {len(all_bboxes_haar[frame_id])}")

cap.release()

# IoU для Haar
all_ious_haar = []
for frame_id in range(301):
    pred_boxes = all_bboxes_haar.get(frame_id, [])
    gt_boxes = [ann["bbox"] for ann in gt_annotations.get(frame_id, [])]
    if not gt_boxes:
        continue
    all_ious_haar.extend(match_bboxes(pred_boxes, gt_boxes, iou_thresh=0.3))

mean_iou_haar = sum(all_ious_haar) / len(all_ious_haar)
matched_haar = sum(1 for x in all_ious_haar if x > 0)

print(f"\n=== Haar Cascade ===")
print(f"Matched: {matched_haar} ({100*matched_haar/len(all_ious_haar):.1f}%)")
print(f"Mean IoU: {mean_iou_haar:.4f}")

print(f"\n=== Итоговое сравнение методов ===")
print(f"{'Метод':<20} {'Matched':>10} {'Mean IoU':>10}")
print(f"{'Frame Diff':<20} {'56.4%':>10} {'0.3436':>10}")
print(f"{'YOLOv8n':<20} {f'{100*matched_yolo/len(all_ious_yolo):.1f}%':>10} {f'{mean_iou_yolo:.4f}':>10}")
print(f"{'Haar Cascade':<20} {f'{100*matched_haar/len(all_ious_haar):.1f}%':>10} {f'{mean_iou_haar:.4f}':>10}")

Кадр 0/301, bbox'ов: 4
Кадр 50/301, bbox'ов: 7
Кадр 100/301, bbox'ов: 4
Кадр 150/301, bbox'ов: 9
Кадр 200/301, bbox'ов: 7
Кадр 250/301, bbox'ов: 8
Кадр 300/301, bbox'ов: 9

=== Haar Cascade ===
Matched: 1724 (47.3%)
Mean IoU: 0.2089

=== Итоговое сравнение методов ===
Метод                   Matched   Mean IoU
Frame Diff                56.4%     0.3436
YOLOv8n                   58.3%     0.4932
Haar Cascade              47.3%     0.2089


In [19]:
import json

W, H = 1920, 1080  # размер кадров из XML

coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "car"},
        {"id": 2, "name": "minivan"}
    ]
}

ann_id = 0
for frame_id in range(301):
    coco["images"].append({
        "id": frame_id,
        "file_name": f"{frame_id:05d}.jpg",
        "width": W,
        "height": H
    })
    for (x, y, w, h) in all_bboxes_yolo.get(frame_id, []):
        coco["annotations"].append({
            "id": ann_id,
            "image_id": frame_id,
            "category_id": 1,  # всё как car, YOLO не различает car/minivan
            "bbox": [x, y, w, h],
            "area": w * h,
            "iscrowd": 0
        })
        ann_id += 1

with open("extracted_annotations_coco.json", "w") as f:
    json.dump(coco, f, indent=2)

print(f"Сохранено: {ann_id} аннотаций по {len(coco['images'])} кадрам")

Сохранено: 2303 аннотаций по 301 кадрам


In [21]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import json
import cv2
import numpy as np

class CarDataset(Dataset):
    def __init__(self, coco_path, frames_dir, frame_ids):
        with open(coco_path) as f:
            self.coco = json.load(f)

        self.frames_dir = frames_dir
        self.frame_ids = frame_ids  # список frame_id для этого сплита

        # Индекс: frame_id → список аннотаций
        self.ann_index = {}
        for ann in self.coco["annotations"]:
            fid = ann["image_id"]
            if fid not in self.ann_index:
                self.ann_index[fid] = []
            self.ann_index[fid].append(ann)

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        frame_id = self.frame_ids[idx]

        # Загружаем кадр
        img_path = f"{self.frames_dir}/{frame_id:05d}.jpg"
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Аннотации
        anns = self.ann_index.get(frame_id, [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w > 5 and h > 5:  # фильтр совсем мелких
                boxes.append([x, y, x + w, y + h])  # xyxy для torchvision
                labels.append(ann["category_id"])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor(frame_id)}
        return img_tensor, target

# Train/val split — 80/20
all_ids = list(range(301))
train_ids = all_ids[:240]
val_ids   = all_ids[240:]

train_dataset = CarDataset("extracted_annotations_coco.json", "frames", train_ids)
val_dataset   = CarDataset("extracted_annotations_coco.json", "frames", val_ids)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)} кадров, Val: {len(val_dataset)} кадров")

Train: 240 кадров, Val: 61 кадров


In [22]:
train_ids = all_ids[:210]       # 70% — 210 кадров
val_ids   = all_ids[210:255]    # 15% — 45 кадров
test_ids  = all_ids[255:]       # 15% — 46 кадров

train_dataset = CarDataset("extracted_annotations_coco.json", "frames", train_ids)
val_dataset   = CarDataset("extracted_annotations_coco.json", "frames", val_ids)
test_dataset  = CarDataset("extracted_annotations_coco.json", "frames", test_ids)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 210, Val: 45, Test: 46


In [ ]:
# Модель
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=3)  # bg + car + minivan

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# Early stopping
best_val_loss = float("inf")
patience = 5
patience_counter = 0

train_losses, val_losses = [], []

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    # --- TRAIN ---
    model.train()
    epoch_loss = 0
    for imgs, targets in train_loader:
        imgs    = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)

    # --- VAL ---
    model.train()  # в режиме train чтобы получать losses
    val_loss = 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs    = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(imgs, targets)
            val_loss += sum(loss_dict.values()).item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    scheduler.step()

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✓ Лучшая модель сохранена (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping на эпохе {epoch+1}")
            break

# График потерь
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses,   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png")
plt.show()

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\KatyaSigma/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:28<00:00, 5.96MB/s] 


Устройство: cpu
Epoch 01/20 | Train Loss: 0.4883 | Val Loss: 0.3867
  ✓ Лучшая модель сохранена (val_loss=0.3867)
